# Extrator de pedidos de delivery com LoRA

No Colab, selecione **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**. Execute as células em ordem. A comparação usa o mesmo Qwen2.5-0.5B-Instruct e os mesmos 30 pedidos de teste para as duas versões. Os dados são sintéticos.


In [ ]:
from pathlib import Path
import subprocess
import sys

projeto = Path('/content/extrator-pedidos')
arquivos = {'README.md': '# Extrator de pedidos de delivery em português\n\nUm pedido como “Até R$ 45 com entrega, sem camarão e em 30 minutos” vira uma estrutura que um serviço pode validar e usar para filtrar um cardápio.\n\n```json\n{\n  "orcamento_max_centavos": 4500,\n  "tempo_max_min": 30,\n  "restricoes": [],\n  "ingredientes_excluidos": ["camarao"]\n}\n```\n\nO contrato usa quatro dos campos do Sugestão da Casa. Restrições cadastradas, como `sem_lactose`, têm um campo separado de ingredientes específicos. Preferências vagas não viram números. Um limite ausente vira `null` e listas ausentes ficam vazias.\n\n## Reproduzir\n\nUse o [notebook para Google Colab](notebooks/Extrator-LoRA-Colab.ipynb) com GPU ou siga as instruções abaixo para rodar localmente.\n\nCrie um ambiente com Python 3.11 e instale o PyTorch 2.8.0 adequado à sua máquina. O exemplo abaixo usa CUDA 12.6.\n\n```sh\npython -m pip install torch==2.8.0 --index-url https://download.pytorch.org/whl/cu126\npython -m pip install -r requirements.txt\npython -m unittest -v\npython experimento.py --passos 120 --seed 42\n```\n\nO script baixa a revisão fixada do [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct). Se você já tiver essa revisão em disco, informe `--modelo-local caminho/do/modelo`. O script usa CUDA quando disponível e CPU nos demais casos.\n\n## Como a comparação funciona\n\nO gerador oferece 360 exemplos sintéticos reproduzíveis. O piloto usa 120 desses exemplos, com um exemplo por passo. O teste contém 30 pedidos escritos à mão, com construções diferentes, correções, ingredientes múltiplos e limites em texto. A auditoria impede frases duplicadas ou compartilhadas entre os conjuntos. Ambos são sintéticos.\n\nO treinamento atualiza apenas adaptadores LoRA nas projeções `q_proj` e `v_proj`. O cálculo da perda ignora os tokens da instrução e do pedido. Para comparar, o script desliga e religa o adaptador, com a mesma instrução, conjunto de teste e geração determinística nas duas versões.\n\n**JSON válido** exige exatamente quatro campos, tipos corretos, ausência de chaves repetidas e valores compatíveis com o contrato. **Acerto completo** exige que os quatro campos coincidam com o gabarito. A ordem das listas não afeta o resultado. Saídas inválidas contam como erro em todas as métricas.\n\nO script salva as métricas, as respostas brutas por pedido e o adaptador em `resultados/`. Ele não salva os pesos do modelo base.\n\n## Resultado do primeiro treino\n\nForam 120 passos, com um pedido por passo e semente 42. O treino levou 21,32 s em uma Tesla T4 no Colab. O tempo não inclui download do modelo ou avaliação.\n\n| Versão | JSON dentro do contrato | Quatro campos corretos |\n| --- | ---: | ---: |\n| Qwen base, resposta original | 0/30 | 0/30 |\n| Qwen base, removendo a cerca Markdown | 28/30 | 1/30 |\n| Qwen ajustado com LoRA | 27/30 | 15/30 |\n\nO modelo base cercou todas as respostas com Markdown. A segunda linha remove só essa formatação, mantém as demais regras do contrato e ajuda a comparar a extração. O LoRA acertou orçamento em 25 pedidos, prazo em 24, restrições em 20 e ingredientes excluídos em 22, contando saídas inválidas como erro.\n\nAinda houve seis pedidos em que a saída passou pelo contrato mas omitiu pelo menos uma restrição cadastrada. Por exemplo, no pedido “Sem lactose e sem amendoim. Se tiver cebola, também não serve”, o LoRA excluiu cebola e deixou a lista de restrições vazia. O adaptador serve para pesquisa e demonstração. Antes de integrá-lo ao recomendador, precisamos avaliar restrições em um conjunto maior e manter a validação das exigências do pedido.\n\nPara conferir o piloto, consulte as [métricas](resultados/metricas.json) e as [respostas por pedido](resultados/predicoes.jsonl). O notebook também recarrega o adaptador em um novo processo e testa o pedido de exemplo.\n\n## Auditoria com 40 pedidos inéditos\n\nRecarreguei o mesmo adaptador e comparei suas respostas com as do Qwen base e com o interpretador por palavras-chave do Sugestão da Casa. O conjunto tem cinco grupos de oito casos, sem sobreposição de texto com o treino ou com o teste inicial.\n\n| Abordagem | JSON válido | Pedido correto | Restrição omitida em JSON válido | Saída inválida entre pedidos com restrição |\n| --- | ---: | ---: | ---: | ---: |\n| Regras do Sugestão da Casa | 40/40 | 18/40 | 6/26 | 0/26 |\n| Qwen base, resposta original | 1/40 | 0/40 | 0/26 | 26/26 |\n| Qwen base, sem cerca Markdown | 35/40 | 8/40 | 8/26 | 4/26 |\n| Qwen ajustado com LoRA | 34/40 | 14/40 | 15/26 | 6/26 |\n\nO denominador 26 corresponde aos pedidos com pelo menos uma restrição cadastrada no gabarito. A auditoria conta pedidos afetados e separa omissões em saídas válidas de respostas que não podem ser usadas. Assim, zero omissões do Qwen base na resposta original não indica que ele preservou restrições: todas as 26 saídas relevantes eram inválidas.\n\nEm ingredientes específicos, o LoRA omitiu um em saída válida em 1 dos 10 pedidos com exclusões, contra 8 das regras. Também inventou pelo menos uma exclusão em 14 dos 40 pedidos, contra 2 das regras. Em um pedido direto com “sem lactose”, o LoRA devolveu uma lista vazia e as regras preservaram a restrição.\n\nA decisão foi manter o LoRA como experimento de portfólio. Nesta amostra, as regras existentes acertaram mais pedidos completos e perderam menos restrições cadastradas. Para reproduzir, obtenha o [Sugestão da Casa na revisão avaliada](https://github.com/Mateus-Tieppo/sugestao-da-casa/tree/8902aa0c20e47f667beb27a93a5326938b91b391), instale `pydantic>=2,<3` e execute:\n\n```sh\npython auditoria.py --backend caminho/sugestao-da-casa/backend\n```\n\nUse `--somente-regras` para rodar sem carregar o Qwen. A pasta [resultados/auditoria](resultados/auditoria) contém as predições, as métricas e o SHA-256 do adaptador avaliado.\n\n## Experimentar com um pedido próprio\n\nUse o adaptador salvo pelo treino ou baixe `adapter_config.json` e `adapter_model.safetensors` do [Hugging Face](https://huggingface.co/Tieppo/extrator-pedidos-lora) para a pasta `resultados/adaptador`. Depois, passe um pedido pela linha de comando:\n\n```sh\npython inferir.py --adaptador resultados/adaptador "Até R$ 45 com entrega, sem camarão e em 30 minutos"\n```\n\nO comando escreve apenas o JSON validado em caso de sucesso. Se a saída do modelo não seguir o contrato, termina com código 2. Omitir `--adaptador` usa o modelo base.\n\n## Limitações\n\nOs dois conjuntos têm 30 e 40 casos sintéticos e exploram distribuições diferentes. Os números não estimam desempenho com pedidos reais. Uma integração precisa verificar a interpretação do pedido e manter os filtros de cardápio antes de oferecer uma recomendação.\n', 'MODEL_CARD.md': '---\nbase_model: Qwen/Qwen2.5-0.5B-Instruct\nbase_model_relation: adapter\nlibrary_name: peft\nlanguage:\n  - pt\ntags:\n  - lora\n  - information-extraction\n  - structured-output\n---\n\n# Extrator de pedidos de delivery em português\n\nAdaptador LoRA experimental para o Qwen2.5-0.5B-Instruct. Converte um pedido em quatro campos: valor máximo em centavos, prazo máximo em minutos, restrições cadastradas e ingredientes excluídos.\n\nO treino usou 120 exemplos de um conjunto de 360 pedidos sintéticos, com semente 42, rank 8 e projeções `q_proj` e `v_proj`. A revisão do modelo base foi `7ae557604adf67be50417f59c2c2f167def9a775`. O ajuste levou 21,32 s em uma Tesla T4.\n\nEm 30 pedidos sintéticos escritos à mão e separados do treino, o modelo ajustado produziu 27 saídas dentro do contrato e acertou os quatro campos em 15 pedidos. O modelo base produziu 0 saídas dentro do contrato no formato original. Após remover apenas a cerca Markdown das respostas, ficou com 28 saídas válidas e 1 pedido completamente correto.\n\nEm seis pedidos, o adaptador devolveu JSON válido e omitiu pelo menos uma restrição cadastrada. O teste pequeno e sintético limita a interpretação dos resultados. O adaptador é experimental e não é adequado para decidir sozinho sobre pedidos com restrições alimentares.\n\nEm uma segunda auditoria com 40 pedidos inéditos, o adaptador acertou os quatro campos em 14 casos. O Qwen base, após remover a cerca Markdown, acertou 8. O interpretador por regras do Sugestão da Casa acertou 18. Dos 26 pedidos com restrição cadastrada, o LoRA omitiu pelo menos uma em 15 saídas válidas e retornou formato inválido em outros 6. Esses resultados motivaram a decisão de manter o modelo fora do recomendador.\n\n## Usar o adaptador\n\nBaixe `adapter_config.json` e `adapter_model.safetensors` para a mesma pasta. O script `inferir.py` do projeto carrega a revisão fixada do Qwen, aplica o adaptador e valida a resposta.\n\n```sh\npython inferir.py --adaptador caminho/do/adaptador "Até R$ 45 com entrega, sem camarão e em 30 minutos"\n```\n\nO adaptador foi treinado sobre o [Qwen2.5-0.5B-Instruct na revisão indicada acima](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/tree/7ae557604adf67be50417f59c2c2f167def9a775), distribuído pelo Qwen sob Apache 2.0.\n', 'requirements.txt': 'accelerate==1.6.0\npeft==0.15.2\ntransformers==4.51.3\n', '.gitignore': '.tools/\n.venv/\n__pycache__/\n*.pyc\nresultados/adaptador/\nresultados/regras-local/\n', 'contrato.py': 'import json\nimport re\n\n\nCAMPOS = ("orcamento_max_centavos", "tempo_max_min", "restricoes", "ingredientes_excluidos")\nRESTRICOES = {\n    "sem_lactose", "sem_gluten", "vegetariano", "vegano", "sem_amendoim", "sem_frutos_do_mar", "sem_ovo",\n}\n\n\ndef _sem_duplicatas(pares):\n    resultado = dict(pares)\n    if len(resultado) != len(pares):\n        raise ValueError("Chave JSON repetida")\n    return resultado\n\n\ndef parse(texto):\n    dados = json.loads(texto, object_pairs_hook=_sem_duplicatas)\n    if not isinstance(dados, dict) or set(dados) != set(CAMPOS):\n        raise ValueError("O JSON deve conter exatamente os quatro campos")\n    for campo in CAMPOS[:2]:\n        valor = dados[campo]\n        if valor is not None and (type(valor) is not int or valor <= 0):\n            raise ValueError(f"{campo} deve ser um inteiro positivo ou null")\n    for campo in CAMPOS[2:]:\n        itens = dados[campo]\n        if not isinstance(itens, list) or any(not isinstance(item, str) for item in itens):\n            raise ValueError(f"{campo} deve ser uma lista de strings")\n        if len(set(itens)) != len(itens):\n            raise ValueError(f"{campo} tem valores repetidos")\n        dados[campo] = sorted(itens)\n    if not set(dados["restricoes"]) <= RESTRICOES:\n        raise ValueError("Restrição fora do catálogo")\n    if any(not re.fullmatch(r"[a-z]+(?: [a-z]+)*", item) for item in dados["ingredientes_excluidos"]):\n        raise ValueError("Ingredientes devem usar minúsculas, sem acentos")\n    return {campo: dados[campo] for campo in CAMPOS}\n\n\ndef avaliar(esperados, respostas):\n    if not esperados or len(esperados) != len(respostas):\n        raise ValueError("Cada pedido deve ter exatamente uma resposta")\n    validos = exatos = 0\n    acertos = dict.fromkeys(CAMPOS, 0)\n    for esperado, resposta in zip(esperados, respostas, strict=True):\n        try:\n            previsto = parse(resposta)\n        except ValueError:\n            continue\n        validos += 1\n        exatos += previsto == esperado\n        for campo in CAMPOS:\n            acertos[campo] += previsto[campo] == esperado[campo]\n    return {"total": len(esperados), "validos": validos, "exatos": exatos, "acertos_por_campo": acertos}\n', 'dados.py': 'import csv\nimport json\nimport random\nfrom pathlib import Path\n\nfrom contrato import CAMPOS, parse\n\n\ndef alvo(preco=None, tempo=None, restricoes=(), ingredientes=()):\n    dados = dict(zip(CAMPOS, (preco, tempo, list(restricoes), list(ingredientes)), strict=True))\n    return parse(json.dumps(dados))\n\n\ndef carregar_teste():\n    caminho = Path(__file__).parent / "dados" / "teste.tsv"\n    with caminho.open(encoding="utf-8", newline="") as arquivo:\n        linhas = csv.DictReader(arquivo, delimiter="\\t")\n        return [\n            (\n                l["pedido"],\n                alvo(\n                    int(l["centavos"]) if l["centavos"] else None,\n                    int(l["minutos"]) if l["minutos"] else None,\n                    l["restricoes"].split(",") if l["restricoes"] else (),\n                    l["ingredientes"].split(",") if l["ingredientes"] else (),\n                ),\n            )\n            for l in linhas\n        ]\n\n\nRESTRICOES = (\n    ("sem glúten", "sem_gluten"),\n    ("zero lactose", "sem_lactose"),\n    ("tem que ser vegano", "vegano"),\n    ("opção vegetariana", "vegetariano"),\n    ("sem amendoim", "sem_amendoim"),\n    ("sem frutos do mar", "sem_frutos_do_mar"),\n    ("não pode ter ovo", "sem_ovo"),\n)\nINGREDIENTES = (\n    ("cebola", "cebola"), ("alho", "alho"), ("camarão", "camarao"),\n    ("coentro", "coentro"), ("tomate", "tomate"), ("pimentão", "pimentao"),\n    ("azeitona", "azeitona"), ("presunto", "presunto"), ("cogumelo", "cogumelo"),\n    ("pimenta", "pimenta"), ("carne de porco", "carne de porco"), ("maionese", "maionese"),\n)\nPREFERENCIAS = (\n    "tô pensando num prato leve", "quero almoçar", "hoje queria comida caseira",\n    "se tiver algo quente melhor", "quero algo baratinho", "me indica um lanche",\n    "alguma coisa pra jantar", "me surpreenda", "quero algo gostoso e rápido",\n)\n\n\ndef treino(total=360, seed=42):\n    rng = random.Random(seed)\n    exemplos = {}\n    while len(exemplos) < total:\n        partes = []\n        preco = tempo = None\n        restricoes, ingredientes = [], []\n        if rng.random() < 0.65:\n            reais = rng.randrange(18, 86)\n            centavos = rng.choice((0, 0, 0, 50, 90, 75))\n            preco = reais * 100 + centavos\n            valor = f"{reais},{centavos:02d}" if centavos else str(reais)\n            partes.append(rng.choice(("até R$ {}", "no máximo {} reais", "limite de R$ {}")).format(valor))\n        if rng.random() < 0.60:\n            if rng.random() < 0.14:\n                frase, tempo = rng.choice((("em até meia hora", 30), ("dentro de uma hora", 60)))\n            else:\n                tempo = rng.randrange(15, 76)\n                frase = rng.choice(("até {} min", "em no máximo {} minutos", "não pode passar de {} minutos")).format(tempo)\n            partes.append(frase)\n        if rng.random() < 0.59:\n            grupos = rng.sample(RESTRICOES, 2 if rng.random() < 0.21 else 1)\n            if {r for _, r in grupos} == {"vegano", "vegetariano"}:\n                grupos = grupos[:1]\n            partes.extend(frase for frase, _ in grupos)\n            restricoes = [r for _, r in grupos]\n        if rng.random() < 0.56:\n            itens = rng.sample(INGREDIENTES, 2 if rng.random() < 0.38 else 1)\n            nomes = " nem ".join(frase for frase, _ in itens)\n            partes.append(rng.choice(("sem {}", "não coloca {}", "tira {} por favor")).format(nomes))\n            ingredientes = [item for _, item in itens]\n        if partes and rng.random() < 0.55:\n            rng.shuffle(partes)\n        if not partes or rng.random() < 0.65:\n            partes.insert(0, rng.choice(PREFERENCIAS))\n        pedido = rng.choice((", ", " e ", ". ")).join(partes)\n        exemplos[pedido] = alvo(preco, tempo, restricoes, ingredientes)\n    return list(exemplos.items())\n\n\ndef verificar_particoes(treino_, teste_):\n    def chaves(linhas):\n        return {" ".join(texto.casefold().split()) for texto, _ in linhas}\n\n    a, b = chaves(treino_), chaves(teste_)\n    if a & b or len(a) != len(treino_) or len(b) != len(teste_):\n        raise ValueError("Pedido repetido dentro ou entre conjuntos")\n', 'experimento.py': 'import argparse\nimport json\nimport random\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport torch\nfrom peft import LoraConfig, get_peft_model\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nfrom contrato import avaliar\nfrom dados import carregar_teste, treino, verificar_particoes\n\n\nMODELO = "Qwen/Qwen2.5-0.5B-Instruct"\nREVISAO = "7ae557604adf67be50417f59c2c2f167def9a775"\nPROMPT = """Extraia somente requisitos explícitos de um pedido em português.\nRetorne apenas um objeto JSON com quatro chaves:\n- orcamento_max_centavos: valor máximo em centavos, ou null. R$ 25,90 vira 2590.\n- tempo_max_min: prazo máximo em minutos, ou null. Meia hora vira 30.\n- restricoes: lista com itens aplicáveis dentre sem_lactose, sem_gluten, vegetariano,\n  vegano, sem_amendoim, sem_frutos_do_mar, sem_ovo.\n- ingredientes_excluidos: lista de ingredientes específicos recusados, minúsculos e sem acentos.\nUse lista vazia quando não houver itens. Não repita na lista de ingredientes uma\nrestrição que já está na lista de restrições. Preferências vagas como barato ou rápido\nnão definem números. Considere a última exigência quando houver uma correção."""\n\n\ndef mensagens(pedido):\n    return [{"role": "system", "content": PROMPT}, {"role": "user", "content": pedido}]\n\n\ndef carregar_modelo(modelo_local=None):\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    origem = str(modelo_local) if modelo_local else MODELO\n    versao = {"local_files_only": True} if modelo_local else {"revision": REVISAO}\n    tokenizer = AutoTokenizer.from_pretrained(origem, padding_side="left", **versao)\n    model = AutoModelForCausalLM.from_pretrained(\n        origem, torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,\n        use_safetensors=True, **versao\n    ).to(device)\n    return model, tokenizer, device\n\n\ndef tokenizar_treino(tokenizer, pedido, alvo):\n    prefixo = tokenizer.apply_chat_template(mensagens(pedido), add_generation_prompt=True)\n    resposta = {"role": "assistant", "content": json.dumps(alvo, ensure_ascii=False, separators=(",", ":"))}\n    tokens = tokenizer.apply_chat_template(mensagens(pedido) + [resposta])\n    if tokens[:len(prefixo)] != prefixo or len(tokens) <= len(prefixo):\n        raise RuntimeError("O template do tokenizer não isolou a resposta do assistente")\n    return tokens, [-100] * len(prefixo) + tokens[len(prefixo):]\n\n\ndef ajustar(base, tokenizer, exemplos, device, passos, seed, destino):\n    model = get_peft_model(\n        base,\n        LoraConfig(task_type="CAUSAL_LM", revision=REVISAO, r=8, lora_alpha=16, lora_dropout=0.05,\n                   target_modules=["q_proj", "v_proj"]),\n    )\n    model.print_trainable_parameters()\n    model.train()\n    otim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=4e-4)\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n    rng = random.Random(seed)\n    ordem = rng.sample(exemplos, k=len(exemplos))\n    inicio = perf_counter()\n    for i in range(passos):\n        if i and i % len(ordem) == 0:\n            rng.shuffle(ordem)\n        texto, alvo = ordem[i % len(ordem)]\n        tokens, labels = tokenizar_treino(tokenizer, texto, alvo)\n        entradas = torch.tensor([tokens], device=device)\n        gabarito = torch.tensor([labels], device=device)\n        otim.zero_grad(set_to_none=True)\n        loss = model(input_ids=entradas, labels=gabarito, use_cache=False).loss\n        if not torch.isfinite(loss):\n            raise RuntimeError(f"Loss inválida no passo {i + 1}")\n        scaler.scale(loss).backward()\n        scaler.unscale_(otim)\n        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)\n        scaler.step(otim)\n        scaler.update()\n        if i == 0 or (i + 1) % 15 == 0 or i + 1 == passos:\n            print(f"Passo {i + 1}/{passos}: loss={loss.item():.4f}", flush=True)\n    model.save_pretrained(destino, safe_serialization=True)\n    return model, round(perf_counter() - inicio, 2)\n\n\n@torch.inference_mode()\ndef prever(model, tokenizer, exemplos, device, lote=5):\n    model.eval()\n    saidas = []\n    inicio = perf_counter()\n    for i in range(0, len(exemplos), lote):\n        conversas = [mensagens(pedido) for pedido, _ in exemplos[i:i + lote]]\n        entradas = tokenizer.apply_chat_template(\n            conversas, add_generation_prompt=True, padding=True, return_dict=True, return_tensors="pt"\n        ).to(device)\n        tokens = model.generate(**entradas, max_new_tokens=120, do_sample=False)\n        novos = tokens[:, entradas["input_ids"].shape[1]:]\n        saidas.extend(tokenizer.batch_decode(novos, skip_special_tokens=True))\n        print(f"  {len(saidas)}/{len(exemplos)} previsões", flush=True)\n    return saidas, round(perf_counter() - inicio, 2)\n\n\ndef main():\n    cli = argparse.ArgumentParser()\n    cli.add_argument("--passos", type=int, default=120)\n    cli.add_argument("--seed", type=int, default=42)\n    cli.add_argument("--modelo-local", type=Path, help="Cópia local da revisão fixada")\n    cli.add_argument("--saida", type=Path, default=Path("resultados"))\n    args = cli.parse_args()\n    if args.passos <= 0:\n        cli.error("--passos deve ser positivo")\n    args.saida.mkdir(parents=True, exist_ok=True)\n    treino_, teste_ = treino(seed=args.seed), carregar_teste()\n    verificar_particoes(treino_, teste_)\n    torch.manual_seed(args.seed)\n    model, tokenizer, device = carregar_modelo(args.modelo_local)\n    print(f"Dispositivo: {device}. Treino: {len(treino_)}. Teste: {len(teste_)}.", flush=True)\n    # Guardar a referência publicável mesmo quando os pesos vierem de uma cópia local.\n    model.config._name_or_path = MODELO\n    model.name_or_path = MODELO\n    model, segundos_treino = ajustar(\n        model, tokenizer, treino_, device, args.passos, args.seed, args.saida / "adaptador"\n    )\n    print("Avaliação do modelo base", flush=True)\n    with model.disable_adapter():\n        base, segundos_base = prever(model, tokenizer, teste_, device)\n    print("Avaliação do modelo ajustado", flush=True)\n    lora, segundos_lora = prever(model, tokenizer, teste_, device)\n    esperados = [alvo for _, alvo in teste_]\n    relatorio = {\n        "modelo": MODELO, "revisao": REVISAO, "seed": args.seed,\n        "dispositivo": str(device), "torch": torch.__version__,\n        "gpu": torch.cuda.get_device_name() if device.type == "cuda" else None,\n        "exemplos_de_treino_disponiveis": len(treino_),\n        "passos_treino": args.passos, "segundos_treino": segundos_treino,\n        "base": {**avaliar(esperados, base), "segundos_inferencia": segundos_base},\n        "lora": {**avaliar(esperados, lora), "segundos_inferencia": segundos_lora},\n    }\n    (args.saida / "metricas.json").write_text(json.dumps(relatorio, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    with (args.saida / "predicoes.jsonl").open("w", encoding="utf-8") as arq:\n        for (pedido, esperado), b, l in zip(teste_, base, lora, strict=True):\n            arq.write(json.dumps({"pedido": pedido, "esperado": esperado, "base": b, "lora": l}, ensure_ascii=False) + "\\n")\n    print(json.dumps(relatorio, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'inferir.py': 'import argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nfrom contrato import parse\n\n\ndef main():\n    cli = argparse.ArgumentParser(description="Extrai os critérios de um pedido e valida o JSON.")\n    cli.add_argument("pedido")\n    cli.add_argument("--adaptador", type=Path, help="Pasta local do adaptador LoRA. Omitir usa o modelo base.")\n    cli.add_argument("--modelo-local", type=Path, help="Cópia local da revisão fixada do Qwen.")\n    args = cli.parse_args()\n    if not args.pedido.strip():\n        cli.error("O pedido não pode ser vazio.")\n    if args.adaptador and not (args.adaptador / "adapter_config.json").is_file():\n        cli.error("A pasta do adaptador deve conter adapter_config.json.")\n\n    import torch\n    from peft import PeftModel\n\n    from experimento import carregar_modelo, mensagens\n\n    model, tokenizer, device = carregar_modelo(args.modelo_local)\n    if args.adaptador:\n        model = PeftModel.from_pretrained(model, str(args.adaptador), local_files_only=True)\n    model.eval()\n    entrada = tokenizer.apply_chat_template(\n        mensagens(args.pedido), add_generation_prompt=True, return_dict=True, return_tensors="pt"\n    ).to(device)\n    with torch.inference_mode():\n        tokens = model.generate(**entrada, max_new_tokens=120, do_sample=False)\n    resposta = tokenizer.decode(tokens[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True)\n    try:\n        dados = parse(resposta)\n    except ValueError as erro:\n        print(f"Saída fora do contrato: {erro}", file=sys.stderr)\n        return 2\n    print(json.dumps(dados, ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'test_contrato.py': 'import unittest\n\nfrom contrato import parse, avaliar\n\n\nVALIDO = \'{"orcamento_max_centavos":4500,"tempo_max_min":30,"restricoes":["sem_gluten"],"ingredientes_excluidos":["cebola","camarao"]}\'\nVAZIO = \'{"orcamento_max_centavos":null,"tempo_max_min":null,"restricoes":[],"ingredientes_excluidos":[]}\'\n\n\nclass ContratoTest(unittest.TestCase):\n    def test_ordena_ingredientes_sem_mudar_seu_significado(self):\n        self.assertEqual(parse(VALIDO)["ingredientes_excluidos"], ["camarao", "cebola"])\n\n    def test_rejeita_chave_extra_duplicada_e_valores_fora_do_contrato(self):\n        invalidos = (\n            VALIDO.replace(\'"tempo_max_min":30\', \'"tempo_max_min":true\'),\n            VALIDO.replace(\'"tempo_max_min":30\', \'"tempo_max_min":-30\'),\n            VALIDO.replace(\'"tempo_max_min":30\', \'"tempo_max_min":30.0\'),\n            VALIDO.replace(\'"tempo_max_min":30\', \'"tempo_max_min":30,"tempo_max_min":30\'),\n            VALIDO.replace(\'"tempo_max_min":30\', \'"tempo_max_min":30,"extra":3\'),\n            VALIDO.replace(\'"tempo_max_min":30,\', \'\'),\n            VALIDO.replace(\'"sem_gluten"\', \'"picante"\'),\n            VALIDO.replace(\'"cebola","camarao"\', \'"cebola","cebola"\'),\n            "Aqui está: " + VALIDO,\n        )\n        for entrada in invalidos:\n            with self.subTest(entrada=entrada), self.assertRaises(ValueError):\n                parse(entrada)\n\n    def test_erros_de_formato_contam_no_denominador(self):\n        resultado = avaliar([parse(VALIDO), parse(VAZIO)], [VALIDO, "erro"])\n        self.assertEqual(resultado["total"], 2)\n        self.assertEqual(resultado["validos"], 1)\n        self.assertEqual(resultado["exatos"], 1)\n        self.assertEqual(resultado["acertos_por_campo"]["tempo_max_min"], 1)\n\n    def test_nao_ignora_previsoes_ausentes(self):\n        with self.assertRaises(ValueError):\n            avaliar([parse(VALIDO)], [])\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'dados/teste.tsv': 'pedido\tcentavos\tminutos\trestricoes\tingredientes\nCom a taxa junto, meu teto é R$ 45. Dispenso camarão. Preciso receber dentro de meia hora.\t4500\t30\t\tcamarao\nPode ser qualquer coisa, só não quero pimentão nem azeitona.\t\t\t\tpimentao,azeitona\nSe chegar depois de 37 minutos não serve.\t\t37\t\t\nTenho R$ 32,90 pra gastar no total.\t3290\t\t\t\nÉ pra uma pessoa celíaca. O prato precisa ser sem glúten e sem ovo.\t\t\tsem_gluten,sem_ovo\t\nNão faço questão de ser vegetariano. Pode ter carne, mas tira o alho.\t\t\t\talho\nSem pressa e sem limite de preço. Só queria algo quentinho.\t\t\t\t\nTenho intolerância à lactose. Não põe tomate.\t\t\tsem_lactose\ttomate\nEu pago até cinquenta reais com a entrega; quero algo vegano.\t5000\t\tvegano\t\nR$ 29,75 é tudo que eu tenho. Chega em 25 minutos ou menos?\t2975\t25\t\t\nNão me manda nada com amendoim. Também não gosto de cebola.\t\t\tsem_amendoim\tcebola\nÉ pra alguém vegetariano. Pimentão pode; coentro, não.\t\t\tvegetariano\tcoentro\nEscolhe um jantar caprichado pra mim. Se for barato, melhor.\t\t\t\t\nTenho uma hora no máximo e queria evitar qualquer fruto do mar.\t\t60\tsem_frutos_do_mar\t\nMeu orçamento não passa de 44 reais; se chegar em até 22 min, perfeito.\t4400\t22\t\t\nJá com entrega, não posso gastar mais de trinta reais. Nem ovo, nem glúten.\t3000\t\tsem_gluten,sem_ovo\t\nTô com pressa mas não tenho um horário certo. Maionese não rola.\t\t\t\tmaionese\nAceito alho. Exclui só cogumelo e presunto.\t\t\t\tcogumelo,presunto\nEu falei 40 antes, mas hoje o máximo é R$ 36.\t3600\t\t\t\nOntem esperava 20 minutos; hoje posso aguardar até 55 minutos.\t\t55\t\t\nSem lactose e sem amendoim. Se tiver cebola, também não serve.\t\t\tsem_lactose,sem_amendoim\tcebola\nQuero uma refeição vegana e dispenso pimenta, por gentileza.\t\t\tvegano\tpimenta\nQuero gastar no máximo R$ 42,50. Tenho 40 minutos e não como frutos do mar.\t4250\t40\tsem_frutos_do_mar\t\nPode ser demorado. Não me importo com glúten ou lactose.\t\t\t\t\nManda uma comida de até R$ 27 sem azeitona e sem alho, por favor.\t2700\t\t\tazeitona,alho\nNão curto carne de porco, mas frango pode. O teto é 57 reais.\t5700\t\t\tcarne de porco\nMeu intervalo termina em 28 min, então esse é o máximo. Nada com ovo.\t\t28\tsem_ovo\t\nGostaria de um prato vegetariano até R$ 62, com entrega em até 35 minutos e sem tomate.\t6200\t35\tvegetariano\ttomate\nSe não tiver coentro, melhor: por favor tira coentro. Até R$ 21,90.\t2190\t\t\tcoentro\nPrecisa ser vegano e sem amendoim. Se custar até R$ 53 e chegar em 33 min no máximo, serve.\t5300\t33\tsem_amendoim,vegano\t\n', 'auditoria.py': 'import argparse\nimport csv\nimport json\nimport re\nimport sys\nfrom collections import Counter\nfrom pathlib import Path\n\nfrom contrato import CAMPOS, parse\nfrom dados import alvo, carregar_teste, treino, verificar_particoes\n\n\ndef carregar_auditoria():\n    caminho = Path(__file__).parent / "dados" / "auditoria.tsv"\n    with caminho.open(encoding="utf-8", newline="") as arquivo:\n        return [\n            (linha["grupo"], linha["pedido"], alvo(\n                int(linha["centavos"]) if linha["centavos"] else None,\n                int(linha["minutos"]) if linha["minutos"] else None,\n                linha["restricoes"].split(",") if linha["restricoes"] else (),\n                linha["ingredientes"].split(",") if linha["ingredientes"] else (),\n            ))\n            for linha in csv.DictReader(arquivo, delimiter="\\t")\n        ]\n\n\ndef tirar_cerca(texto):\n    cerca = re.fullmatch(r"```json\\s*([\\s\\S]*?)\\s*```", texto.strip(), flags=re.I)\n    return cerca[1] if cerca else texto\n\n\ndef resumir(casos, saidas):\n    if len(casos) != len(saidas) or not casos:\n        raise ValueError("Cada caso deve ter uma resposta")\n    cont = Counter(dict.fromkeys((\n        "total", "validos", "exatos", "com_restricao", "com_exclusao",\n        "restricao_omitida_em_saida_valida", "restricao_em_saida_invalida",\n        "restricao_inventada", "ingrediente_omitido_em_saida_valida", "exclusao_em_saida_invalida",\n        "ingrediente_inventado",\n    ), 0))\n    for (_, _, esperado), raw in zip(casos, saidas, strict=True):\n        cont["total"] += 1\n        restricoes, ingredientes = (set(esperado[c]) for c in CAMPOS[2:])\n        cont["com_restricao"] += bool(restricoes)\n        cont["com_exclusao"] += bool(ingredientes)\n        try:\n            obtido = parse(raw)\n        except ValueError:\n            cont["restricao_em_saida_invalida"] += bool(restricoes)\n            cont["exclusao_em_saida_invalida"] += bool(ingredientes)\n            continue\n        cont["validos"] += 1\n        cont["exatos"] += obtido == esperado\n        r, ing = (set(obtido[c]) for c in CAMPOS[2:])\n        cont["restricao_omitida_em_saida_valida"] += bool(restricoes - r)\n        cont["restricao_inventada"] += bool(r - restricoes)\n        cont["ingrediente_omitido_em_saida_valida"] += bool(ingredientes - ing)\n        cont["ingrediente_inventado"] += bool(ing - ingredientes)\n    return dict(cont)\n\n\ndef prever_regras(backend, casos):\n    sys.path.insert(0, str(backend.resolve(strict=True)))\n    from app.demo.interpretador import interpretar\n\n    resultados = []\n    for _, pedido, _ in casos:\n        dados = interpretar(pedido, None).criterios.model_dump(mode="json", include=set(CAMPOS))\n        resultados.append(json.dumps(dados, ensure_ascii=False))\n    return resultados\n\n\ndef main():\n    cli = argparse.ArgumentParser()\n    cli.add_argument("--backend", required=True, type=Path)\n    cli.add_argument("--adaptador", type=Path, default=Path("resultados/adaptador"))\n    cli.add_argument("--saida", type=Path, default=Path("resultados/auditoria"))\n    cli.add_argument("--somente-regras", action="store_true")\n    args = cli.parse_args()\n    casos = carregar_auditoria()\n    pares = [(texto, esperado) for _, texto, esperado in casos]\n    verificar_particoes(treino() + carregar_teste(), pares)\n    saidas = {"regras": prever_regras(args.backend, casos)}\n    if not args.somente_regras:\n        from peft import PeftModel\n        from experimento import carregar_modelo, prever\n\n        if not (args.adaptador / "adapter_config.json").is_file():\n            cli.error("Adaptador não encontrado")\n        modelo, tokenizer, device = carregar_modelo()\n        modelo = PeftModel.from_pretrained(modelo, str(args.adaptador), local_files_only=True)\n        with modelo.disable_adapter():\n            saidas["base_original"], _ = prever(modelo, tokenizer, pares, device)\n        saidas["base_sem_cerca"] = [tirar_cerca(p) for p in saidas["base_original"]]\n        saidas["lora"], _ = prever(modelo, tokenizer, pares, device)\n\n    resumo = {nome: resumir(casos, respostas) for nome, respostas in saidas.items()}\n    args.saida.mkdir(parents=True, exist_ok=True)\n    (args.saida / "metricas.json").write_text(json.dumps(resumo, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    with (args.saida / "predicoes.jsonl").open("w", encoding="utf-8") as arq:\n        for i, (grupo, pedido, esperado) in enumerate(casos):\n            linha = {"grupo": grupo, "pedido": pedido, "esperado": esperado}\n            linha.update({nome: respostas[i] for nome, respostas in saidas.items()})\n            arq.write(json.dumps(linha, ensure_ascii=False) + "\\n")\n    print(json.dumps(resumo, ensure_ascii=False, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'test_auditoria.py': 'import json\nimport unittest\n\nfrom auditoria import carregar_auditoria, resumir, tirar_cerca\nfrom dados import alvo, carregar_teste, treino, verificar_particoes\n\n\nclass AuditoriaTest(unittest.TestCase):\n    def test_separa_omissao_em_json_valido_de_saida_invalida(self):\n        esperado = alvo(restricoes=("sem_ovo",), ingredientes=("cebola",))\n        casos = [("teste", "sem ovo nem cebola", esperado)] * 2\n        respostas = [json.dumps(alvo(restricoes=("sem_gluten",))), "texto"]\n        medidas = resumir(casos, respostas)\n        self.assertEqual(medidas["validos"], 1)\n        self.assertEqual(medidas["com_restricao"], 2)\n        self.assertEqual(medidas["restricao_omitida_em_saida_valida"], 1)\n        self.assertEqual(medidas["restricao_em_saida_invalida"], 1)\n        self.assertEqual(medidas["restricao_inventada"], 1)\n        self.assertEqual(medidas["ingrediente_omitido_em_saida_valida"], 1)\n        self.assertEqual(medidas["exclusao_em_saida_invalida"], 1)\n\n    def test_remove_so_cerca_markdown_completa(self):\n        corpo = json.dumps(alvo())\n        self.assertEqual(tirar_cerca(f"```json\\n{corpo}\\n```"), corpo)\n        frase = f"Resultado: ```json\\n{corpo}\\n```"\n        self.assertEqual(tirar_cerca(frase), frase)\n\n    def test_auditoria_tem_casos_novos_em_cinco_grupos(self):\n        casos = carregar_auditoria()\n        self.assertEqual(len(casos), 40)\n        self.assertEqual(len({g for g, _, _ in casos}), 5)\n        verificar_particoes(treino() + carregar_teste(), [(t, e) for _, t, e in casos])\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'dados/auditoria.tsv': 'grupo\tpedido\tcentavos\tminutos\trestricoes\tingredientes\ndireto\tHoje só peço algo sem lactose, até R$ 48.\t4800\t\tsem_lactose\t\ndireto\tTraz um almoço vegetariano em até 32 minutos.\t\t32\tvegetariano\t\ndireto\tPreciso de uma opção sem glúten por até R$ 38,50.\t3850\t\tsem_gluten\t\ndireto\tMe indica um jantar sem amendoim.\t\t\tsem_amendoim\t\ndireto\tQuero um prato vegano com entrega em até 45 min.\t\t45\tvegano\t\ndireto\tEscolhe um prato sem frutos do mar por até R$ 52.\t5200\t\tsem_frutos_do_mar\t\ndireto\tHoje vou de sanduíche sem ovo.\t\t\tsem_ovo\t\ndireto\tUm almoço sem pimentão, até R$ 34,20.\t3420\t\t\tpimentao\nparafrase\tLactose me faz mal. O que tem no cardápio?\t\t\tsem_lactose\t\nparafrase\tSou celíaco, então glúten fica fora.\t\t\tsem_gluten\t\nparafrase\tTenho alergia a amendoim, consegue achar outra opção?\t\t\tsem_amendoim\t\nparafrase\tOvo me dá alergia. Preciso de um prato que não leve esse ingrediente.\t\t\tsem_ovo\t\nparafrase\tMinha dieta é vegana, mesmo se o prato só levar queijo já não serve.\t\t\tvegano\t\nparafrase\tFrutos do mar estão fora para mim. O orçamento é de R$ 46 no máximo.\t4600\t\tsem_frutos_do_mar\t\nparafrase\tPor favor, retire a cebola e mantenha todos os outros ingredientes.\t\t\t\tcebola\nparafrase\tPara mim camarão é que não dá. Peixe pode.\t\t\t\tcamarao\ncombinacao\tHoje preciso de um prato sem lactose e sem glúten, com entrega em 26 minutos no máximo.\t\t26\tsem_gluten,sem_lactose\t\ncombinacao\tQuero um jantar vegetariano, sem ovo e sem cebola.\t\t\tsem_ovo,vegetariano\tcebola\ncombinacao\tComida vegana e nada de amendoim. Dá para gastar até R$ 49,90.\t4990\t\tsem_amendoim,vegano\t\ncombinacao\tNão como frutos do mar nem ovo. Meu limite é R$ 54 e a espera máxima é de 31 min.\t5400\t31\tsem_frutos_do_mar,sem_ovo\t\ncombinacao\tTem que ser sem glúten. Pode tirar tomate, coentro e alho?\t\t\tsem_gluten\talho,coentro,tomate\ncombinacao\tO pedido não pode ter lactose. Também não quero presunto nem azeitona.\t\t\tsem_lactose\tazeitona,presunto\ncombinacao\tUm prato sem amendoim e sem pimenta. A entrega pode levar no máximo meia hora.\t\t30\tsem_amendoim\tpimenta\ncombinacao\tTira cogumelo e maionese de um lanche vegano. Até R$ 39,50.\t3950\t\tvegano\tcogumelo,maionese\ncontraste\tNão precisa ser vegano. Só peço que seja sem ovo.\t\t\tsem_ovo\t\ncontraste\tGlúten está liberado, mas eu preciso evitar lactose.\t\t\tsem_lactose\t\ncontraste\tEu disse vegetariano, mas mudei de ideia. Pode ter carne; mantenha sem glúten.\t\t\tsem_gluten\t\ncontraste\tMeu amigo é alérgico a amendoim, mas o pedido é só meu e posso comer de tudo.\t\t\t\t\ncontraste\tPode pôr camarão. O que eu não quero é cebola nem alho.\t\t\t\talho,cebola\ncontraste\tPensei em pedir sem ovo, mas o ovo pode ficar. Quero sem amendoim.\t\t\tsem_amendoim\t\ncontraste\tO prato não precisa ser sem glúten, pode ser normal. Só tira pimentão.\t\t\t\tpimentao\ncontraste\tQueijo sem lactose tudo bem. Só preciso que seja vegetariano e sem lactose.\t\t\tsem_lactose,vegetariano\t\ncontrole\tEscolhe uma sopa bem quente. Posso esperar.\t\t\t\t\ncontrole\tQuero um lanche de até R$ 28,90 com entrega em até 24 minutos.\t2890\t24\t\t\ncontrole\tVou querer sobremesa com amendoim, se tiver.\t\t\t\t\ncontrole\tSem pressa para chegar; qualquer prato serve.\t\t\t\t\ncontrole\tQualquer almoço com bastante alho e cebola.\t\t\t\t\ncontrole\tCapricha na pimenta, por favor, e fica abaixo de R$ 50.\t5000\t\t\t\ncontrole\tQueria algo barato e rápido, sem um teto de preço nem prazo fixo.\t\t\t\t\ncontrole\tHoje queria camarão ou peixe e posso esperar no máximo 42 minutos.\t\t42\t\t\n'}
for nome, texto in arquivos.items():
    arquivo = projeto / nome
    arquivo.parent.mkdir(parents=True, exist_ok=True)
    arquivo.write_text(texto, encoding='utf-8')


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Selecione um ambiente com GPU e reinicie a sessão.')
print(torch.__version__, torch.cuda.get_device_name())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=projeto, check=True)
subprocess.run([sys.executable, '-m', 'unittest', '-v'], cwd=projeto, check=True)


In [ ]:
subprocess.run([sys.executable, '-u', 'experimento.py', '--passos', '120', '--seed', '42'], cwd=projeto, check=True)


In [ ]:
import os

amostra = subprocess.run(
    [sys.executable, '-u', 'inferir.py', '--adaptador', 'resultados/adaptador',
     'Até R$ 45 com entrega, sem camarão e em 30 minutos'],
    cwd=projeto, capture_output=True, text=True, encoding='utf-8',
    env={**os.environ, 'PYTHONUTF8': '1'},
)
print(amostra.stdout or amostra.stderr)
if amostra.returncode and 'Saída fora do contrato:' not in amostra.stderr:
    amostra.check_returncode()


In [ ]:
import json
import shutil

resultados = projeto / 'resultados'
metricas = json.loads((resultados / 'metricas.json').read_text(encoding='utf-8'))
for versao in ('base', 'lora'):
    m = metricas[versao]
    print(f'{versao}: JSON válido {m["validos"]}/{m["total"]}; acerto completo {m["exatos"]}/{m["total"]}')
pacote = shutil.make_archive('/content/resultados-extrator', 'zip', resultados)
print('Baixe pelo painel Arquivos do Colab:', pacote)


## Auditoria das restrições

A célula abaixo recarrega o adaptador do primeiro treino e compara três abordagens em 40 pedidos novos. Ela usa a revisão fixa do interpretador do Sugestão da Casa.


In [ ]:
import hashlib

rev = '8902aa0c20e47f667beb27a93a5326938b91b391'
repo = Path('/content/sugestao-regras-8902aa0')
repo.mkdir(exist_ok=True)
subprocess.run(['git', 'init', '-q'], cwd=repo, check=True)
subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'https://github.com/Mateus-Tieppo/sugestao-da-casa.git', rev], cwd=repo, check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', 'FETCH_HEAD'], cwd=repo, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip() == rev
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pydantic>=2,<3'], check=True)
subprocess.run([sys.executable, '-u', 'auditoria.py', '--backend', str(repo / 'backend')], cwd=projeto, check=True)
auditoria = resultados / 'auditoria'
pesos = resultados / 'adaptador' / 'adapter_model.safetensors'
(auditoria / 'proveniencia.json').write_text(json.dumps({
    'regras_commit': rev, 'adaptador_sha256': hashlib.sha256(pesos.read_bytes()).hexdigest(),
}, indent=2), encoding='utf-8')
print((auditoria / 'metricas.json').read_text(encoding='utf-8'))
print(shutil.make_archive('/content/auditoria-resultados', 'zip', auditoria))
